In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

In [ ]:
class Generator(nn.Module):  # Définition de la classe du générateur
    def __init__(self, z_dim=100, channels_img=3, features_g=64):  # Constructeur
        super(Generator, self).__init__()  # Appel du constructeur parent (PyTorch)

        self.net = nn.Sequential(  # On empile les couches dans un modèle séquentiel

            # --------- Étape 1 : bruit → feature map ---------
            nn.ConvTranspose2d(z_dim, features_g * 16, 4, 1, 0),  
            # Transforme le vecteur latent (z_dim) en une petite carte 4x4 avec beaucoup de canaux

            nn.BatchNorm2d(features_g * 16),  
            # Normalise les activations pour stabiliser l'entraînement

            nn.ReLU(True),  
            # Activation non linéaire (favorise valeurs positives)

            # --------- Étape 2 : upsampling ---------
            nn.ConvTranspose2d(features_g * 16, features_g * 8, 4, 2, 1),  
            # Double la taille spatiale (4x4 → 8x8)

            nn.BatchNorm2d(features_g * 8),  
            # Stabilisation et acceleration de l'entrainement

            nn.ReLU(True),  

            # --------- Étape 3 ---------
            nn.ConvTranspose2d(features_g * 8, features_g * 4, 4, 2, 1),  
            # 8x8 → 16x16

            nn.BatchNorm2d(features_g * 4),  
            nn.ReLU(True),

            # --------- Étape 4 ---------
            nn.ConvTranspose2d(features_g * 4, features_g * 2, 4, 2, 1),  
            # 16x16 → 32x32

            nn.BatchNorm2d(features_g * 2),
            nn.ReLU(True),

            # --------- Étape finale : image ---------
            nn.ConvTranspose2d(features_g * 2, channels_img, 4, 2, 1),  
            # 32x32 → 64x64 (image finale)

            nn.Tanh()  
            # Ramène les pixels entre [-1, 1]
        )

    def forward(self, x):  # Définition du passage avant
        return self.net(x)  # Passe les données dans le réseau